# 🔓 Solutions: Model Evaluation - Regression

> **Try first, peek second.** You learn far more by struggling with a problem for ten minutes than by reading its answer for ten seconds. Use these solutions to *compare* your approach, not to replace it. There is often more than one correct way.


In [1]:
import sys
sys.path.append("..")
from course_utils import *
setup()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             mean_absolute_percentage_error)

### 🧩 Problem 6.1: Metrics from scratch &nbsp; ⭐⭐

Implement the four metrics using **only NumPy** (no `sklearn.metrics`): `mae(y, p)`, `mse(y, p)`, `rmse(y, p)` and `r2(y, p)`.
They should accept lists or arrays.

In [2]:
def mae(y, p):
    e = np.asarray(y, dtype=float) - np.asarray(p, dtype=float)
    return np.mean(np.abs(e))

def mse(y, p):
    e = np.asarray(y, dtype=float) - np.asarray(p, dtype=float)
    return np.mean(e ** 2)

def rmse(y, p):
    return np.sqrt(mse(y, p))

def r2(y, p):
    y, p = np.asarray(y, dtype=float), np.asarray(p, dtype=float)
    return 1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2)

print(mae([250, 300, 450, 600, 900], [270, 280, 500, 560, 820]), r2([250, 300, 450, 600, 900], [270, 280, 500, 560, 820]))

42.0 0.9589090909090909


In [3]:
# ✔ Self-check
_y, _p = [250, 300, 450, 600, 900], [270, 280, 500, 560, 820]
check("MAE of the five houses is 42", lambda: abs(mae(_y, _p) - 42) < 1e-9)
check("MSE is 2260", lambda: abs(mse(_y, _p) - 2260) < 1e-9)
check("RMSE is √2260", lambda: abs(rmse(_y, _p) - 2260 ** 0.5) < 1e-9)
check("R² matches sklearn", lambda: abs(r2(_y, _p) - r2_score(_y, _p)) < 1e-9)
_rng = np.random.default_rng(3); _a, _b = _rng.normal(size=100), _rng.normal(size=100)
check("all four match sklearn on random data", lambda: all([
    abs(mae(_a, _b) - mean_absolute_error(_a, _b)) < 1e-9, abs(mse(_a, _b) - mean_squared_error(_a, _b)) < 1e-9,
    abs(r2(_a, _b) - r2_score(_a, _b)) < 1e-9]))

✅ PASSED   MAE of the five houses is 42
✅ PASSED   MSE is 2260
✅ PASSED   RMSE is √2260
✅ PASSED   R² matches sklearn
✅ PASSED   all four match sklearn on random data


True

**🧑‍🏫 Walkthrough.** Each metric is one line once you have the residuals. Writing them yourself removes the mystery: **R²** is simply `1 − (your model's squared error) / (the squared error of always guessing the mean)`. Keep these functions: we'll reuse the idea in the Practical Assignment.

### 🧩 Problem 6.2: One bad apple (outlier sensitivity) &nbsp; ⭐⭐

Below are actual values and predictions for 10 items. Then **one** prediction gets badly wrong (the last item's error becomes **+100**).

Compute `mae_before`, `mae_after`, `rmse_before`, `rmse_after`, and the **growth factors** `mae_factor = mae_after / mae_before` and `rmse_factor = rmse_after / rmse_before`.

In [4]:
actual = np.arange(200, 400, 20, dtype=float)
err = np.array([5, -4, 6, -5, 4, -6, 5, -4, 6, -5], dtype=float)
pred_before = actual + err
err_after = err.copy(); err_after[-1] = 100
pred_after = actual + err_after

mae_before, mae_after = mean_absolute_error(actual, pred_before), mean_absolute_error(actual, pred_after)
rmse_before = np.sqrt(mean_squared_error(actual, pred_before))
rmse_after = np.sqrt(mean_squared_error(actual, pred_after))
mae_factor, rmse_factor = mae_after / mae_before, rmse_after / rmse_before
print(f"MAE : {mae_before:.2f} → {mae_after:.2f}  (×{mae_factor:.1f})")
print(f"RMSE: {rmse_before:.2f} → {rmse_after:.2f}  (×{rmse_factor:.1f})")

MAE : 5.00 → 14.50  (×2.9)
RMSE: 5.06 → 31.99  (×6.3)


In [5]:
# ✔ Self-check
check("MAE grows by roughly 2.5-4.5×", lambda: 2.5 < mae_factor < 4.5)
check("RMSE grows by a larger factor than MAE", lambda: rmse_factor > mae_factor)
check("RMSE grows by more than 5×", lambda: rmse_factor > 5)

✅ PASSED   MAE grows by roughly 2.5-4.5×
✅ PASSED   RMSE grows by a larger factor than MAE
✅ PASSED   RMSE grows by more than 5×


True

**🧑‍🏫 Walkthrough.** One outlier multiplies MAE by ~2.9 but RMSE by ~6.3, because squaring turns a 100-unit miss into 10,000 while the small errors contribute only ~25 each. This is the *practical* meaning of "RMSE is outlier-sensitive": use it when big mistakes are truly worse than many small ones, and MAE when they aren't.

### 🧩 Problem 6.3: Beat the baseline &nbsp; ⭐⭐

On the housing data (split provided), compute on the **test set**:

* `dummy_r2`: R² of `DummyRegressor(strategy="mean")` trained on the training set;
* `lin_r2`: R² of `LinearRegression`;
* `dummy_rmse`, `lin_rmse`: their RMSEs;
* `improvement_pct` = how much lower the linear model's RMSE is than the baseline's, in percent: `100 * (dummy_rmse − lin_rmse) / dummy_rmse`.

In [6]:
housing = load("housing.csv")
X, y = housing.drop(columns="price_k"), housing["price_k"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

dummy = DummyRegressor(strategy="mean").fit(X_train, y_train)
lin = LinearRegression().fit(X_train, y_train)

dummy_pred, lin_pred = dummy.predict(X_test), lin.predict(X_test)
dummy_r2, lin_r2 = r2_score(y_test, dummy_pred), r2_score(y_test, lin_pred)
dummy_rmse = np.sqrt(mean_squared_error(y_test, dummy_pred))
lin_rmse = np.sqrt(mean_squared_error(y_test, lin_pred))
improvement_pct = 100 * (dummy_rmse - lin_rmse) / dummy_rmse
print(f"baseline R² = {dummy_r2:.3f}, linear R² = {lin_r2:.3f};  RMSE {dummy_rmse:.0f} → {lin_rmse:.0f}  ({improvement_pct:.0f}% better)")

baseline R² = -0.000, linear R² = 0.972;  RMSE 183 → 31  (83% better)


In [7]:
# ✔ Self-check
check("baseline R² is about 0 (between −0.05 and 0.05)", lambda: -0.05 < dummy_r2 < 0.05)
check("linear model R² above 0.95", lambda: lin_r2 > 0.95)
check("improvement_pct is correct", lambda: abs(improvement_pct - 100 * (dummy_rmse - lin_rmse) / dummy_rmse) < 1e-9)
check("RMSE dropped by more than 80%", lambda: improvement_pct > 80)

✅ PASSED   baseline R² is about 0 (between −0.05 and 0.05)
✅ PASSED   linear model R² above 0.95
✅ PASSED   improvement_pct is correct
✅ PASSED   RMSE dropped by more than 80%


True

**🧑‍🏫 Walkthrough.** The baseline's test R² is essentially zero (slightly negative is normal; the test mean differs a little from the training mean). "R² = 0.97" is only impressive because we know the floor is 0; and "RMSE cut by ~85%" is a statement anyone can grasp. **Always report a baseline.**

### 🧩 Problem 6.4: Hunt the biggest mistakes &nbsp; ⭐⭐

Train a `LinearRegression` on the housing training set (split provided). On the **test set**, build a DataFrame `worst` containing the **five houses with the largest absolute error**, with columns `area_sqm`, `actual`, `predicted`, `abs_error`, sorted from worst to less bad.

Also store `mean_price_of_worst`: the average *actual* price of those five, and `mean_price_all`: the average actual price in the test set.

In [8]:
housing = load("housing.csv")
X, y = housing.drop(columns="price_k"), housing["price_k"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

pred = LinearRegression().fit(X_train, y_train).predict(X_test)
df = pd.DataFrame({"area_sqm": X_test["area_sqm"], "actual": y_test, "predicted": pred,
                   "abs_error": np.abs(y_test - pred)})
worst = df.sort_values("abs_error", ascending=False).head(5)
mean_price_of_worst, mean_price_all = worst["actual"].mean(), y_test.mean()
print(worst.round(1)); print(round(mean_price_of_worst), "vs", round(mean_price_all))

     area_sqm  actual  predicted  abs_error
830     239.7  1055.9      924.4      131.5
418     174.1   678.8      803.3      124.5
892     170.8   909.1      816.5       92.6
441     229.5   967.1      894.5       72.6
299     150.5   661.7      591.8       69.9
855 vs 476


In [9]:
# ✔ Self-check
_pred = LinearRegression().fit(X_train, y_train).predict(X_test)
_err = np.abs(y_test.values - _pred)
check("five rows", lambda: len(worst) == 5)
check("columns are area_sqm, actual, predicted, abs_error", lambda: list(worst.columns) == ["area_sqm", "actual", "predicted", "abs_error"])
check("first row has the largest absolute error", lambda: abs(worst["abs_error"].iloc[0] - _err.max()) < 1e-6)
check("sorted from worst to less bad", lambda: worst["abs_error"].is_monotonic_decreasing)
check("the worst misses are on pricier-than-average houses", lambda: mean_price_of_worst > mean_price_all)

✅ PASSED   five rows
✅ PASSED   columns are area_sqm, actual, predicted, abs_error
✅ PASSED   first row has the largest absolute error
✅ PASSED   sorted from worst to less bad
✅ PASSED   the worst misses are on pricier-than-average houses


True

**🧑‍🏫 Walkthrough.** The five worst misses average a much higher price than the typical test house: the model's biggest errors are concentrated in the luxury end. Spotting *where* a model fails is the first step to fixing it, and no single summary metric can tell you that.

### 🧩 Problem 6.5: MAE or RMSE? Choose for the business &nbsp; ⭐⭐

Two delivery-time models predict the same 10 deliveries (arrays below). Compute each model's **MAE** and **RMSE** (store as `mae_A, rmse_A, mae_B, rmse_B`).

Then answer with `"A"` or `"B"` in the dictionary `choice`:

* `"typical_customer"`: we care about the *typical* error a customer experiences (every minute counts the same).
* `"avoid_disasters"`: a single very late delivery causes a lost customer, so large misses must be avoided.

In [10]:
actual = np.arange(20, 70, 5, dtype=float)
pred_A = actual + np.array([1, -1, 1, -1, 1, -1, 1, -1, 1, 30], dtype=float)
pred_B = actual + np.array([7, -7, 7, -7, 7, -7, 7, -7, 7, -7], dtype=float)

mae_A, mae_B = mean_absolute_error(actual, pred_A), mean_absolute_error(actual, pred_B)
rmse_A, rmse_B = np.sqrt(mean_squared_error(actual, pred_A)), np.sqrt(mean_squared_error(actual, pred_B))
choice = {"typical_customer": "A", "avoid_disasters": "B"}
print(f"A: MAE {mae_A:.1f}, RMSE {rmse_A:.1f}    B: MAE {mae_B:.1f}, RMSE {rmse_B:.1f}")

A: MAE 3.9, RMSE 9.5    B: MAE 7.0, RMSE 7.0


In [11]:
# ✔ Self-check
check("A has the lower MAE", lambda: mae_A < mae_B)
check("B has the lower RMSE", lambda: rmse_B < rmse_A)
check("typical customer → A", lambda: choice["typical_customer"] == "A")
check("avoid disasters → B", lambda: choice["avoid_disasters"] == "B")

✅ PASSED   A has the lower MAE
✅ PASSED   B has the lower RMSE
✅ PASSED   typical customer → A
✅ PASSED   avoid disasters → B


True

**🧑‍🏫 Walkthrough.** Model A is better for the typical delivery (MAE 3.9 vs 7.0) but has a catastrophic 30-minute miss (RMSE 9.5 vs 7.0). **There is no universally "best" metric: the right one encodes what errors cost your business.** That's why you must understand the problem before choosing a number to optimise.

### 🧩 Problem 6.6: When R² lies (Anscombe's trap) &nbsp; ⭐⭐⭐

Anscombe's dataset II (below) looks like a curve. 

1. Fit a straight-line `LinearRegression` and store its R² in `r2_linear`.
2. Fit a **quadratic** model: `make_pipeline(PolynomialFeatures(2), LinearRegression())`: and store its R² in `r2_quadratic`.
3. Store in `residual_corr` the **absolute correlation between the linear model's residuals and the centred parabola `(x - x.mean())**2`**. *(A value far from 0 means the residuals still contain a pattern the model failed to capture.)*

In [12]:
x = np.array([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5], dtype=float)
y = np.array([9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74])
X2 = x.reshape(-1, 1)

lin = LinearRegression().fit(X2, y)
r2_linear = r2_score(y, lin.predict(X2))
quad = make_pipeline(PolynomialFeatures(2), LinearRegression()).fit(X2, y)
r2_quadratic = r2_score(y, quad.predict(X2))
residual_corr = abs(np.corrcoef(y - lin.predict(X2), (x - x.mean()) ** 2)[0, 1])
print(f"linear R² = {r2_linear:.3f}, quadratic R² = {r2_quadratic:.4f}, residual vs parabola correlation = {residual_corr:.2f}")

linear R² = 0.666, quadratic R² = 1.0000, residual vs parabola correlation = 1.00


In [13]:
# ✔ Self-check
check("straight-line R² is only about 0.67", lambda: abs(r2_linear - 0.666) < 0.01)
check("quadratic R² is nearly perfect (> 0.99)", lambda: r2_quadratic > 0.99)
check("the linear residuals are strongly tied to the parabola (|corr| > 0.9)", lambda: residual_corr > 0.9)

✅ PASSED   straight-line R² is only about 0.67
✅ PASSED   quadratic R² is nearly perfect (> 0.99)
✅ PASSED   the linear residuals are strongly tied to the parabola (|corr| > 0.9)


True

**🧑‍🏫 Walkthrough.** A respectable-looking R² of 0.67 hides the fact that the *shape* is wrong: the linear model's residuals are almost *perfectly* correlated with a parabola (the model is literally telling you the shape it's missing. That's the power of residual analysis. Note also the caveat: with 11 points a quadratic fitting this well tells us the data is curved) not that quadratics are the "true law" of every future x.